In [0]:
CREATE OR REPLACE TABLE employee (
  emp_id      INT         NOT NULL,
  emp_name    STRING      NOT NULL,
  manager_id  INT,
  department  STRING,
  salary      DECIMAL(10, 2)
);

INSERT INTO employee 
VALUES  (1,'A',NULL,'IT',90000),
        (2,'B',1,'IT',80000),
        (3,'C',1,'HR',70000),
        (4,'D',2,'IT',60000),
        (5,'E',2,'IT',50000),
        (6,'F',3,'HR',40000);

In [0]:
WITH RECURSIVE emp_hierarchy AS (
  -- Anchor: top-level employees (no manager)
  SELECT emp_id,
         emp_name,
         manager_id,
         department,
         1 AS level,
         CAST(emp_name AS STRING) AS hierarchy_path
  FROM employee
  WHERE manager_id IS NULL

  UNION ALL

  -- Recursive: direct reports of the previous level
  SELECT e.emp_id,
         e.emp_name,
         e.manager_id,
         e.department,
         h.level + 1,
         CONCAT(h.hierarchy_path, ' -> ', e.emp_name)
  FROM employee e
  INNER JOIN emp_hierarchy h ON e.manager_id = h.emp_id
)
SELECT level,
       emp_id,
       LPAD('', (level - 1) * 4, ' ') || emp_name AS employee,
       manager_id,
       department,
       hierarchy_path
FROM emp_hierarchy
ORDER BY hierarchy_path;

In [0]:
SELECT * FROM employee;

In [0]:
select department,median(salary) as median_salary from employee group by department;

In [0]:
select *,median(salary) over(partition by department) as median_salary from employee;

In [0]:
CREATE OR REPLACE TABLE purchse (
  order_id   INT,
  sale_date  DATE
);

INSERT INTO purchse VALUES
  (1, DATE'2024-01-10'),
  (1, DATE'2024-03-15'),
  (1, DATE'2024-06-20'),
  (2, DATE'2024-02-01'),
  (2, DATE'2024-02-25'),
  (3, DATE'2024-05-05');



In [0]:
SELECT order_id,
       sale_date,
       MIN(sale_date) OVER (PARTITION BY order_id) AS first_purchase_date,
       LEAD(sale_date) OVER (PARTITION BY order_id ORDER BY sale_date) AS next_purchase_date,
       DATEDIFF(
         LEAD(sale_date) OVER (PARTITION BY order_id ORDER BY sale_date),
         sale_date
       ) AS days_between_purchases
FROM purchse
ORDER BY order_id, sale_date;

In [0]:
SELECT order_id,sale_date,
MIN(sale_date) OVER(ORDER BY sale_date) as first_purchase_date,
LEAD(sale_date) OVER(order by sale_date) as next_purchase_date,
DATEDIFF(LEAD(sale_date) OVER(ORDER BY sale_date),sale_date) as days_between_purchases
FROM purchse
order by order_id,sale_date;

In [0]:
SELECT order_id
FROM purchse
WHERE date_format(sale_date, 'yyyy-MM') IN ('2024-01', '2024-02')
GROUP BY order_id
HAVING COUNT(DISTINCT date_format(sale_date, 'yyyy-MM')) = 2;

In [0]:
WITH total_spending AS (
    SELECT order_id,
           SUM(amount) AS total_Amount,
           AVG(amount) AS avg_smount
    FROM sales
    GROUP BY order_id
)
SELECT order_id, total_Amount
FROM total_spending
WHERE total_Amount > avg_smount;